In [0]:
CREATE OR REFRESH STREAMING LIVE TABLE fuel_project_dev.gold.fact_fuel_transactions AS
SELECT 
  t.fill_id,
  t.station_id,
  date_format(t.start_time, 'yyyyMMdd') as date_key,
  hour(t.start_time) as hour_key,
  t.start_time,
  t.end_time,
  timestampdiff(SECOND, t.start_time, t.end_time) as transaction_duration_sec,
  t.fuel_type,
  t.fuel_volume,
  t.payment_type,
  r.fuel_cost,
  t.fuel_volume * r.fuel_cost as revenue,
  current_timestamp() as processed_at
FROM STREAM(fuel_project_dev.silver.fuel_transactions) t
LEFT JOIN fuel_project_dev.silver.fuel_rates r
  ON t.station_id = r.fuel_station_id
  AND t.fuel_type = r.fuel_type
  AND t.start_time >= r.start_datetime
  AND t.start_time < r.end_datetime;